### GlycoMSParser 0.4
demonstration version
This version shows workflow and provides 1 sample file for testing
- Please mind that this tools CURRENTLY only supports "permethylated N-glycan in Positive mode", we will add more supported types and file formats in the future.

<img src=".\GlycoMSP_scheme.jpg" alt="Overview of GlycoMSP" title="GlycoMSP overview" />

***
In our pipeline, we're set to deal with raw files (Thermo Fisher), which requires certain dlls for conversion.
***
<div class="alert alert-block alert-warning">
<b>If you are not running this code on a Windows machine with Thermo library installed, 
please run the "Peakextractor.py" (FilterMS2.py in 0.4, will be integrated in 0.6) on supporting machine and start the analysis from importing csv.</b>
</div>


In [1]:
#GlycoMSParser 0.3
#Import cell
GlycoMSP_version = 0.3
GlycoMSP_date = 20231007
GlycoMSP_build = "manv2"

In [2]:
#GlycoMSParser 0.3
GlycoMSP_version_extractor = 0.2
GlycoMSPinit = True #To indicate cells below if the read process has been completed

#File conversion cell

try:
    from pymsfilereader import MSFileReader
except:
    print("Please run this code on a Windows-based OS with pymsfilereader and Thermo library installed")
#debug
def debug_readraw(debug=False):
    rawfile.Close()
    print("raw file has been closed successfully.")
#end of debug function

#init

try:
    print("GlycoMSP uses tkinter to get raw files") #change this to "to read GlycoMSP project dat, csv in supported formats and ms files" once dev is finished
    from tkinter import Tk
    from tkinter.filedialog import askopenfilename
    Tk().withdraw()
    filename = askopenfilename() #restrict to raw file, with text indications
    rawfile = MSFileReader(filename)
    #print('File choosed', rawfile.Version())
    rawname = rawfile.GetFileName()
    print('The raw file you selected is: ', rawname)
except ImportError:
    raise ImportError('Please install tkinter to enable windows-supported file selection')

if not rawfile:
    try:
        rawfile = MSFileReader("G:\zf_sPerMeOG_intestine.raw")
        print('No tkinter detected. Select default raw file')
        print('Debug: raw file version: ', rawfile.Version())
        rawname = rawfile.GetFileName()
        print('GetFileName: ', rawname)
    except:
        print("Raw file read failed. Please check if the file path is correct.")

print("After importing raw file run the debug to break file handling")
#comment the debug functions below to avoid exit of file handling
#-----------------------------------------------#
#debug_readraw(debug=True)

GlycoMSP uses tkinter to get raw files
The raw file you selected is:  G:/zf_sPerMeNG_brain.raw
After importing raw file run the debug to break file handling


#### Source: DetermineMSLevel.py
Original file last updated: 2022/10/25
Functions included:
- findingMSlevel()  v20221025.1
    -  Which indicates MS Level of "certain" spectrum no. (Not included in this demo)
- ExtractMSlevel()  v20221025.1
    -  Extract MS info to csv, manually
- AutoExtMSlevel()  **in development**
    -  Do the extraction process automatically
#### TODO: Complete development and rename it to spectra_extractor.py

#### Source: trytolistoutheaders.py
Original file last updated: 2023/3/7
Functions included:
- calcproton()  v ????
    -  Calculates molecule mass in protonated form [H+] (inherited from comparepeaklist.py)
- extractor not in a function: v ???? **in development**
    -  Extract MS info to csv, manually
- AutoExtMSlevel()  **in development**
    -  Do the extraction process automatically
#### TODO: Complete development and rename it to spectra_extractor.py

In [3]:
from collections import namedtuple
import time
import glob
import pandas as pd
import os
import pathlib

#copied from comparepeaklist.py
def calcproton(isolatedmass, charge):
    if abs(charge) == 1 :
        return isolatedmass
    elif charge >= 2:
        conv = (isolatedmass * charge - (charge - 1) * 1.00784)
        return conv
    elif charge == 0:
        return 0
    elif charge < -1:
        negconv = (isolatedmass * charge + (charge - 1) * 1.00784)
        return negconv
    

def peak_extractor(rawfile, auto = True, debug = False):

    print(f'This raw file has {rawfile.GetNumSpectra()} spectra')
    #define number of spectra to parse (TODO: make time exclusion convertible and need to be added to record file.)
    if not auto:
        scan_number = int(input('enter spectrum no you want to summarize'))
        maxn = int(rawfile.GetNumSpectra())
    else:
        scan_number = int(rawfile.GetNumSpectra())
        maxn = scan_number

    #init variables for storing parsed data
    ms1list, ms2list, ms3list, errlist = {}, {}, {}, {}
    ms1fromms2 = []
    b, c = [], []  #b for MS2, c for MS3
    header = ('entry no','MS1scan no', 'MS1Isolation mass', 'MS1monoIsomass','chargeState','in [H+]',
          'intensity','Structure', 'MS2 Scan no', 'peaklist')
    b.append(header)
    ms3header =  ('entry no','MS3scan no', 'MS2Isolation mass', 'MS2monoIsomass', 'MS2 Scan no', 'peaklist')
    c.append(ms3header)
    MS2peaklist = namedtuple('MS2peaklist', ('dMass', 'dIntensity'))
    MS3peaklist = namedtuple('MS3peaklist', ('dMass', 'dIntensity'))
    ms2count, ms3count = 1, 1, 

    #development area and exception prevention
    if debug:
        print("[debug] need to create debug logs when debug is set to True.")
        print("[debug]Also need a readable log that records needed information and the operation can be reproduced when loading that record.")
        print("[debug add debug log and record log in debug block]")
        print("[debug]: start calculate time spent for converting data")
        perf_esti1 = time.time()
    
    if scan_number > maxn:
        print(r"you're attempting to run a number more than the spectra this file has.")
        scan_number = maxn
        print('set scan_number to', maxn)
    #
    
    #perform file conversion in defined spectra range
    for i in range(scan_number):
        j = i+1
        if i > maxn:
            break
        elif rawfile.GetMSOrderForScanNum(j) == 1:
            #ms1list[i] = j
            pass
            #shooud I keep this? the list is totally referred from MS2
        elif rawfile.GetMSOrderForScanNum(j) == 2:
            peaklist = MS2peaklist((rawfile.GetLabelData(j)[0][0]), (rawfile.GetLabelData(j)[0][1]))
            isolationmass = rawfile.GetPrecursorInfoFromScanNum(j)[1]
            chargestate = rawfile.GetPrecursorInfoFromScanNum(j)[2]
            inHmass = calcproton(isolationmass,chargestate)
            a = (ms2count,
                rawfile.GetPrecursorInfoFromScanNum(j)[3],#parentScanNo, MS1
                rawfile.GetPrecursorInfoFromScanNum(j)[0],#Isolation mass
                isolationmass,#monoIsomass
                chargestate,#chargeState
                inHmass, #calculate calcproton(isolatedmass, charge)
                'ext from peak list',
                'structure na',
                j, #MS2scan no
                peaklist
                )
            b.append(a)
            ms2count +=1
        elif rawfile.GetMSOrderForScanNum(j) == 3:
            ms3list[i] = j
            peaklist = MS3peaklist((rawfile.GetMassListFromScanNum(j)[0][0]), (rawfile.GetMassListFromScanNum(j)[0][1]))
            d = (ms3count,
                rawfile.GetPrecursorInfoFromScanNum(j)[3],#parentScanNo
                rawfile.GetPrecursorInfoFromScanNum(j)[0],#Isolation mass
                rawfile.GetPrecursorInfoFromScanNum(j)[1],#monoIsomass
                j, #MS2scan no
                peaklist #surely will have error
                )
            c.append(d)
            ms3count +=1
        else:
            errlist[i] = j        
        i+=1
    print("Extraction finished.")
    rawfile.Close()

    if debug:
        extractiontime = time.time() - perf_esti1
        print("[debug]Raw file has been closed.")
        print("[debug]Preparing information for writing...")
        print(f"[debug]Time spent for extraction: {int(extractiontime)} seconds.")

    timestamp = time.strftime("%Y%m%d-%H%M%S") #datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

    #export MS2 spectra file in csv format
    save_ms2filename= input("Please enter the file name or leave it blank to generate a filename with datetime")
    if len(save_ms2filename) == 0:
        fileext = rawname + ' MS2 summary from ' + str(scan_number) + " at " + timestamp + '.csv'
    else:
        fileext = rawname + save_ms2filename + " at " + timestamp + ".csv"
    print(f"Filename output:{fileext}")
    perf_esti2 = time.time()
    with open(fileext, 'wt') as g:
        print("writing to csv...")
        for q in range(len(b)):
            print('\t'.join(map(str, (b[q]))), file = g)
    if debug:
        ms2writetime = time.time() - perf_esti2
        print("[debug]Finished writing MS2 information")
        print(f"[debug]Time spent for extraction: {int(ms2writetime)} seconds.")

    #export MS3 spectra file in csv format
    timestamp = time.strftime("%Y%m%d-%H%M%S") 
    save_ms3filename= input("Please enter the MS3 file name or leave it blank to generate a filename with datetime")
    if len(save_ms3filename) ==0 :
        fileext = rawname + ' MS3 summary from ' + str(scan_number) + " at " + timestamp + '.csv'
    else:
        fileext = rawname + save_ms3filename + " at " + timestamp + ".csv"
    print(f"Filename output:{fileext}")
    perf_esti3 = time.time()
    with open(fileext, 'wt') as h:
        print("writing to csv...")
        for q in range(len(c)):
            print('\t'.join(map(str, (c[q]))), file = h)
    if debug:
        ms3writetime = time.time() - perf_esti3
        print("[debug]Finished writing MS3 information")
        print(f"[debug]Time spent for extraction: {int(ms3writetime)} seconds.")


    #export log files...
    if debug:
        print("Write logs")
        print(f"file name: {fileext}")
        print(f"datetime (latest): {timestamp}")
        print(f"file name: {rawname}")
        print("file path is not possible in current context.")
        print(f"scan number set in this experiment: {scan_number}")
        print(f"GlycoMSP main version: {GlycoMSP_version}")
        #info in logs: datetime, filename, file path, scan number, version of GlycoMSP and EACH components
    
    #report errors
    print('err list')
    for key,value in errlist.items():
        print(value)

    print("dump finished")




def peak_exthandler(rawfile, filetype=None, debug=False):
    if filetype == "raw":
        peak_extractor(rawfile, auto= True, debug=debug)#or debug=debug can work
    elif filetype == "mzML":
        print("mzML file will be supported in v1.1")
    else:
        print("unsupported format")


def runextractor(debug=False):
    if GlycoMSPinit:
        try:
            import glob
            import pandas as pd
            import os
            import pathlib
            import numpy as np
            import time  #for calculating efficiency
            #from collections import namedtuple
            import warnings
            warnings.simplefilter(action='ignore', category=FutureWarning) ###to supress warning
        except:
            print("Missing essential package for following analysis") #try to list out the package missing
        if rawfile:
            print("Start raw file extration process...")
            filetype="raw"
            peak_exthandler(rawfile, filetype, debug)

runextractor(debug=True)

Start raw file extration process...
This raw file has 44218 spectra
[debug] need to create debug logs when debug is set to True.
[debug]Also need a readable log that records needed information and the operation can be reproduced when loading that record.
[debug add debug log and record log in debug block]
[debug]: start calculate time spent for converting data
Extraction finished.
[debug]Raw file has been closed.
[debug]Preparing information for writing...
[debug]Time spent for extraction: 288 seconds.
Please enter the file name or leave it blank to generate a filename with datetime
Filename output:G:/zf_sPerMeNG_brain.raw MS2 summary from 44218 at 20231009-203347.csv
writing to csv...
[debug]Finished writing MS2 information
[debug]Time spent for extraction: 4 seconds.
Please enter the MS3 file name or leave it blank to generate a filename with datetime
Filename output:G:/zf_sPerMeNG_brain.raw MS3 summary from 44218 at 20231009-222454.csv
writing to csv...
[debug]Finished writing MS3 i

In [13]:
#please rerun the file to evaluate the time spent (on remote lab PC)
'''
Start raw file extration process...
This raw file has 44218 spectra
[debug] need to create debug logs when debug is set to True.
[debug]Also need a readable log that records needed information and the operation can be reproduced when loading that record.
[debug add debug log and record log in debug block]
[debug]: start calculate time spent for converting data
Extraction finished.
[debug]Raw file has been closed.
[debug]Preparing information for writing...
[debug]Time spent for extraction: 288 seconds.
Please enter the file name or leave it blank to generate a filename with datetime
Filename output:G:/zf_sPerMeNG_brain.raw MS2 summary from 44218 at 20231009-203347.csv
writing to csv...
[debug]Finished writing MS2 information
[debug]Time spent for extraction: 4 seconds.
Please enter the MS3 file name or leave it blank to generate a filename with datetime
Filename output:G:/zf_sPerMeNG_brain.raw MS3 summary from 44218 at 20231009-222454.csv
writing to csv...
[debug]Finished writing MS3 information
[debug]Time spent for extraction: 0 seconds.
Write logs
file name: G:/zf_sPerMeNG_brain.raw MS3 summary from 44218 at 20231009-222454.csv
datetime (latest): 20231009-222454
file name: G:/zf_sPerMeNG_brain.raw
file path is not possible in current context.
scan number set in this experiment: 44218
GlycoMSP main version: 0.3
err list
dump finished
'''

Read csv and calculate, fill missed information
From FilterMS2_2023.py
In this part we are going to calculate relative intensity for those ions we selected
There are 2 methods will be available in the future
- A: calculate only glycan fragments (fast), which need users to select ions and take highest signal in selected ion set as 100%
- B: calculate ALL peaks, while we are considering if we should remove peaks under a threshold to reduce data size and pre-filter those (possibly) noises before further processing

In [ ]:
import csv
#import math
from decimal import Decimal, getcontext, ROUND_HALF_UP
from collections import namedtuple
#import re
import pandas as pd
import comparepeaklist
#import numpy as np

list1 = (344.1698)
roundedlist = []
roundedlist2 = []
tmplist = []
MS2peaklist = namedtuple('MS2peaklist', ('dMass', 'dIntensity'))

LISTforbrain = [344.1547, 374.1653, 376.1966, 406.2072, 432.2071, 450.2334, 464.2490, 580.2964, 793.3808, 825.4227] #probably, digged from code
LISTforintestine = [246.1336, 260.1492, 303.1438, 335.1700 ,344.1704, 374.1810 ,376.1966, 406.2072, 432.2071, 450.2334,
                     464.2490, 539.2698, 580.2964, 621.3229, 638.3382, 793.3808, 825.4227, 842.4380]

LISTforOvary = [246.1336, 260.1492, 344.1704, 374.1810 ,376.1966, 406.2072, 432.2071, 450.2334, 464.2490, 505.2756, 580.2964, 621.3229, 638.3382,
                651.3335, 793.3808, 825.4227, 842.4380, 866.4492, 896.4598]

#New function in future
def listsetcompare(lista, listb, mode="und"):
    try:
        alist = set(lista)
        blist = set(listb)
    except:
        #consider to add more validation method in this part
        print("[invalid list]The list you included has one or more same values.")
    if mode == "und":
        print("The calculation method isn't defined")
    elif mode == "AND":
        andlist = alist.intersection(blist)
        #do intersection of 2 list
        print("[Return]Intersection of 2 lists")
        return andlist
    elif mode == "uni":
        print("Union of 2 lists")
        unilist = alist.union(blist)
        #do intersection of 2 list
        print("[Return]Union of 2 lists")
        return unilist
    
#Key input one by one (very slow)
def setcomparelist():
    comparelist = []
    while True:
        a = input("enter fragments for searching")
        try:
            comparelist.append(float(a))
        except:
            print("The one you entered isn't a number")
            break
    return comparelist

#Select from a set of ions (in dev)
def setcomparelistfromitem():
    comparelist = []
    #pseudocode
    #list out a dict (fragment name, mass) and allow
    #adding multiple entries at one time
    #delete added fragment after adding (allow adding until user exit)
    #return the tuple of both peak mass (rounded 0.01) and float intensity
    # as a dict or (m, i) or in separated list (better for future design but unreadable by manwork
    # get the whole list info and add simple annotation (referring previous research results)
    #question: cant recognize correctly. See glypick results and extract certain spectrum for testing

#The one reads csv generated by 
def readms2csvpdreg(comparelist=None):
    #put the file path within the same directory with this file
    df = pd.read_csv('zf_sPerMeNG_ovary.raw MS2 summary from 44510 at 20231007-235302.csv', sep='\t') # zf_sPerMeNG_intestine MS2 summary from 48127 at 20230611-182546 20230611-194107 predictedcomp.csv#'debug1.csv the MS2 summary from44218.csv'
    #remove unmaned index
    df.drop(columns=df.columns[0], axis=1, inplace=True) 
    print(df.dtypes)
    #hitspectrumlist = []
    err = 0
    notfound = 0
    hits = 0
    #ms2scanno= 0
    subdf = []
    #tmphitreturnlist = []
    ppm = comparepeaklist.ppmcalculation()
    #cc = []
    #ncc = []
    tmpdf = pd.DataFrame()
    listinfo = True
    for i in range(len(df)):
        #initialize the placeholder for input peak information (to be parsed)
        inputlist = []
        inputintensitylist = []
        #parse peaklist
        #load no. i row's peaklist of the dataframe
        tmp = df.loc[i, "peaklist"]
        #the peak intensity and peak mass are included as well.
        inputlist1 = tmp.split('(')[2].split(')')[0].split(',') #peakmass
        d = tmp.split('(')[3].split(')')[0].split(',') #peakintensity
        #print('inputlist =',type(inputlist), inputlist1, 'intensity=',type(d), d)
        #print('going to modify the number')
        for j in range(len(inputlist1)):
            #should be 0.0001 or adjustable in the future
            k = Decimal((inputlist1[j])).quantize(Decimal("0.001"),rounding=ROUND_HALF_UP) 
            k = float(k)
            l = Decimal((d[j])).quantize(Decimal("0.001"),rounding=ROUND_HALF_UP)
            l = float(l)
            inputlist.append(k)
            inputintensitylist.append(l)
        #loss one OHCH3= 32.0419
        #call user input and if no vaild input, set it to default list
        if comparelist is None:
            #here is pre-defined list [(Neu5Ac-OMe B ion, 344.17), (Neu5Gc-OMe B ion, 374.18), (Neu5AcB ion, 376.20), (Neu5Gc B ion, 406.21), (LacNAc-OMe B ion, 432.22),
            #(LacNAc-BY ion, 450.23), (LacNAc-B ion, 464.25), (Neu5AcHex B ion, 580.30), (SiaLacNAc-OMe B ion, 793.38), (SiaLacNAc B ion, 825.42)]
            comparelist = [344.17, 374.18, 376.20, 406.21, 432.22, 450.23, 464.25, 580.30, 793.38, 825.42] #default one
        if listinfo is True:
            print('compare list=', comparelist)
            listinfo = False #let it only prints once
        result = comparepeaklist.comparepeaklistppm(inputlist, comparelist, ppm, inputintensitylist)
        if result[0] > 0:
            df.loc[i, "in [H+]"] = comparepeaklist.calcproton(df.loc[i, "MS1Isolation mass"],df.loc[i, "chargeState"])
            #ms2scan = df.loc[i, "MS2 Scan no"]
            #hitspectrumlist.append([ms2scan, result[0], result[1], result[3], result[4]])
            hits+=1
            #before append calculate the charge conversion
            #protonateass = comparepeaklist.calcproton(df.loc[i, "MS1Isolation mass"], df.loc[i, "chargeState"])
            #I think I should add them back at thise stage
            #And theoretical mass
            #And prediction of composition if prediction flag is set to True)
            addinfo = pd.Series([result[1], result[3], result[4]], index=['hitpeaklist', 'selectedpeakintensity','normalizedselectedin']) #, result[5], 'maxintensity'
            tmpdf = pd.concat([df.loc[i], addinfo], axis = 0)
            subdf.append(tmpdf)
            #debug1 = int(df.loc[i, "MS2 Scan no"])
            #if debug1 == int(14857):    #test why 376 and the relative intensity value is wrong
            #    print(type(addinfo), type(subdf))
            #    print(addinfo, subdf)
            #    break
        elif result[0] == 0:
            #print('no hits')
            notfound+=1
        else:
            print('unexpected error')
            err+=1
            #do nothing
        #need to catch the returned value and decide if this spectra is ok
    print('hits', hits, 'notfound', notfound, 'err', err) #'hits', hitspectrumlist)
    #w = pd.DataFrame()
    #subdf.to_csv("ms2filteredtotallistwithHtest.csv", index=False)

    #define new file name?
    with open('ovary.csv', 'w', newline='') as zz: #prevois file name: ms2filteredtotallistwithH_NGintestine20230612_50ppm.csv
        roww = csv.writer(zz)
        roww.writerow(["entry no", "MS1scan no", "MS1Isolation mass", "MS1monoIsomass", "chargeState", "in [H+]", "intensity", "Structure", "MS2 Scan no", "peaklist", "hitpeaklist", "selectedpeakintensity", "normalizedselectedintensity"])
        roww.writerows(subdf)
    print("Finished exporting")
    #return hitspectrumlist
    #with open('ms2searchsummary.csv', 'w', newline='') as z:
    #    roww = csv.writer(z)
    #    roww.writerows(hitspectrumlist)


#I think it was a test function
def readms2csvpd():
    df = pd.read_csv('the MS2 summary from350.csv', sep='\t')
    print(df.dtypes)
    for i in range(len(df)):
        print(df.loc[i, "entry no"], df.loc[i, "peaklist"])
        #parse peaklist
        tmp = df.loc[i, "peaklist"]
        tmp = tmp.replace('(', '!').replace(')', '!').split('!')
        #drop named tuple value
        print(tmp[0],'\<zero \> first', tmp[1])
        #tmp = tmp[2]
        print(type(tmp), tmp)
        peaklist = tmp[2:-2]
        print(peaklist)
    #df1 = df["peaklist"]
    print(df)


readms2csvpdreg(LISTforOvary)

filterresultreader.py
v20221122
Able to read csv in correct format and is supposed to carry the file completion & modification manually